In [3]:
# MBE in reward function
# Cell 2: Imports
import re
import torch
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
from src.mbe import mbe_reverse_gram

In [4]:
# Cell 3: Load and prepare GSM8K dataset
dataset = load_dataset("openai/gsm8k", "main")

def extract_gold_answer(answer_text: str) -> str:
    """Extract the numeric answer after ####."""
    match = re.search(r"####\s*(.+)", answer_text)
    if match:
        return match.group(1).strip().replace(",", "")
    return ""

def format_example(example):
    """Convert to chat-style prompt and attach gold answer."""
    example["prompt"] = [{"role": "user", "content": example["question"]}]
    example["gold_answer"] = extract_gold_answer(example["answer"])
    return example

train_dataset = dataset["train"].map(format_example)
test_dataset = dataset["test"].map(format_example)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print(f"Example prompt: {train_dataset[0]['prompt']}")
print(f"Gold answer: {train_dataset[0]['gold_answer']}")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Train: 7473, Test: 1319
Example prompt: [{'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}]
Gold answer: 72


In [5]:
# Cell 4: Define reward functions

def extract_answer_from_completion(text: str) -> str:
    """Parse the final numeric answer from a model completion."""
    # Look for #### pattern first
    match = re.search(r"####\s*([\d,\.\-]+)", text)
    if match:
        return match.group(1).strip().replace(",", "")
    # Fallback: last number in the text
    numbers = re.findall(r"-?[\d,]+\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "")
    return ""

def correctness_reward(completions: list[list[dict]], gold_answer: list[str], **kwargs) -> list[float]:
    """Award +1.0 if the model's final numeric answer matches the gold answer."""
    rewards = []
    for completion, gold in zip(completions, gold_answer):
        text = completion[0]["content"]
        predicted = extract_answer_from_completion(text)
        try:
            correct = float(predicted) == float(gold)
        except (ValueError, TypeError):
            correct = False
        rewards.append(1.0 if correct else 0.0)
    return rewards

def format_reward(completions: list[list[dict]], **kwargs) -> list[float]:
    """Award +0.5 if the response contains #### <number> pattern."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"]
        has_format = bool(re.search(r"####\s*[\d,\.\-]+", text))
        rewards.append(0.5 if has_format else 0.0)
    return rewards

# Quick sanity check
test_comp = [[{"content": "The answer is 2+3=5. #### 5"}]]
print("Correctness:", correctness_reward(test_comp, gold_answer=["5"]))
print("Format:", format_reward(test_comp))

Correctness: [1.0]
Format: [0.5]


In [6]:
@torch.no_grad()
def full_forward(model, input_ids):
    """Single forward pass returning logits and all hidden states."""
    outputs = model(input_ids, output_hidden_states=True, use_cache=False)
    return outputs.logits, outputs.hidden_states

def compute_mbe_trace(hidden_states, prompt_len, patch_size=8, layer=-1):
    h = hidden_states[layer][0, prompt_len:, :]  # (T_comp, D)
    T, D = h.shape
    usable = (T // patch_size) * patch_size
    if usable == 0:
        return torch.tensor([0.0])
    h = h[:usable].reshape(-1, patch_size, D)
    mbe_vals = mbe_reverse_gram(h)
    return mbe_vals

def compute_per_layer_mbe(hidden_states, prompt_len, patch_size=8):
    per_layer = []
    for layer_idx in range(1, len(hidden_states)):
        mbe_vals = compute_mbe_trace(hidden_states, prompt_len, patch_size=patch_size, layer=layer_idx)
        per_layer.append(mbe_vals.mean().item())
    return per_layer


class MBEReward:
    """
    MBE-based reward compatible with TRL's reward function interface.
    Runs a forward pass through the model, computes per-layer MBE on
    completion hidden states, returns mean MBE as a scalar reward.

    Usage:
        mbe_reward = MBEReward(model, tokenizer)
        trainer = GRPOTrainer(reward_funcs=[correctness_reward, mbe_reward], ...)
    """

    def __init__(self, model, tokenizer, patch_size=8, layers=None):
        self.model = model
        self.tokenizer = tokenizer
        self.patch_size = patch_size
        self.layers = layers  # None = all layers (1..N)

    @torch.no_grad()
    def __call__(self, prompts, completions, **kwargs) -> list[float]:
        rewards = []
        device = next(self.model.parameters()).device

        for prompt, completion in zip(prompts, completions):
            # Reconstruct full text via chat template
            prompt_text = self.tokenizer.apply_chat_template(
                prompt, tokenize=False, add_generation_prompt=True
            )
            completion_text = completion[0]["content"]
            full_text = prompt_text + completion_text

            # Tokenize separately to get accurate prompt_len
            prompt_ids = self.tokenizer(prompt_text, return_tensors="pt")["input_ids"]
            full_ids = self.tokenizer(full_text, return_tensors="pt")["input_ids"].to(device)
            prompt_len = prompt_ids.shape[1]

            # Skip if completion too short for a single patch
            comp_len = full_ids.shape[1] - prompt_len
            if comp_len < self.patch_size:
                rewards.append(0.0)
                continue

            # Forward pass to get hidden states
            _, hidden_states = full_forward(self.model, full_ids)

            # Compute MBE per layer
            if self.layers is not None:
                per_layer = []
                for layer_idx in self.layers:
                    mbe_vals = compute_mbe_trace(
                        hidden_states, prompt_len,
                        patch_size=self.patch_size, layer=layer_idx
                    )
                    per_layer.append(mbe_vals.mean().item())
            else:
                per_layer = compute_per_layer_mbe(
                    hidden_states, prompt_len, patch_size=self.patch_size
                )

            # Scalar reward = mean MBE across selected layers
            reward = sum(per_layer) / len(per_layer) if per_layer else 0.0
            rewards.append(reward)

        return rewards

In [8]:
# Cell 5: Training with MBE reward
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")

# MBE reward (uses the training model's hidden states)
mbe_reward = MBEReward(model, tokenizer, patch_size=8)

config = GRPOConfig(
    output_dir="mbe_reward_gsm8k_output",
    num_generations=4,
    max_completion_length=512,
    per_device_train_batch_size=4,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=10,
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward, format_reward, mbe_reward],
    args=config,
    train_dataset=train_dataset,
)

trainer.train()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


ValueError: bf16 mixed precision requires PyTorch >= 1.10 and a supported device.